# Trace quickstart

Generate a deterministic grounded visual-reasoning task, inspect its typed supervision and public execution trace, then replay its reward contract. No credentials or accelerator are required.

[Open the Trace Space](https://huggingface.co/spaces/maveryn/trace) · [Read the documentation](https://maveryn.github.io/trace/)

In [ ]:
%pip install -q \
  "Pillow==12.2.0" "CairoSVG==2.9.0" "tqdm==4.68.4" "PyYAML==6.0.3" \
  "numpy==2.2.6" "scipy==1.15.3" "networkx==3.4.2" "rfc8785==0.1.4" \
  "blake3==1.0.9" "zstandard==0.25.0" "cairocffi==1.7.1" "cffi==2.1.0" \
  "pycparser==3.0" "cssselect2==0.9.0" "tinycss2==1.5.1" \
  "webencodings==0.5.1" "defusedxml==0.7.1" \
  "trace-tasks @ git+https://github.com/maveryn/trace.git@bb7fdd1fc8a0f8a2e3db7efe910a14e81d58feb7"

In [ ]:
import json

from IPython.display import JSON, display
from trace_tasks import generate_task, score_trace_response
from trace_tasks.core.annotation_sanitization import sanitize_trace_payload_for_public_annotation
from trace_tasks.core.reward_contracts import resolve_reward_contract

TASK_ID = "task_geometry__graph_paper__polygon_area_value"
SEED = 42

In [ ]:
sample = generate_task(TASK_ID, seed=SEED, max_attempts=100)
display(sample.image)
print(sample.prompt)

In [ ]:
answer_gt = sample.answer_gt.to_dict()
annotation_gt = sample.annotation_gt.to_dict()
reward_contract = resolve_reward_contract(
    answer_type=sample.answer_gt.type,
    annotation_type=sample.annotation_gt.type,
).to_dict()

display(JSON({
    "answer_gt": answer_gt,
    "annotation_gt": annotation_gt,
    "reward_contract": reward_contract,
}))

In [ ]:
public_trace = sanitize_trace_payload_for_public_annotation(
    sample.trace_payload,
    annotation_gt=sample.annotation_gt,
)
display(JSON(public_trace))

In [ ]:
correct_response = json.dumps({
    "answer": sample.answer_gt.value,
    "annotation": sample.annotation_gt.value,
})
scores = score_trace_response(
    response=correct_response,
    answer_gt=answer_gt,
    annotation_gt=annotation_gt,
    reward_contract=reward_contract,
    image_size=sample.image.size,
)
display(JSON(scores))
assert scores["overall"] == 1.0

## Try another task

Change `TASK_ID` to any value returned by `trace_tasks.list_task_ids()` and choose any non-negative integer `SEED`. The same pair will always reproduce the same validated instance at the pinned code revision.

In [ ]:
from trace_tasks import list_task_ids

print(f"{len(list_task_ids()):,} registered tasks")
print("First five:", *list_task_ids()[:5], sep="\n- ")